# EmoText MZ — Modelos do protótipo

## 1. Configuração

In [ ]:
import os
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate>=1.0', 'datasets'], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)

PASTA_MODELOS = '/content/drive/MyDrive/EmoTextModelos'
PASTA_TEMP = '/content/tmp_treino'
for p in [PASTA_MODELOS, PASTA_TEMP]:
    os.makedirs(p, exist_ok=True)
print('Modelos serão guardados em:', PASTA_MODELOS)

In [ ]:
import json
import re
import shutil
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DISPOSITIVO, torch.cuda.get_device_name(0) if DISPOSITIVO == 'cuda' else '')
assert DISPOSITIVO == 'cuda', 'Active a GPU T4 antes de continuar.'

ID_MODELO = 'FacebookAI/xlm-roberta-base'
SEMENTE = 42
SENTIMENTOS = ['negative', 'neutral', 'positive']
EMOCOES = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
GRELHA_LIMIARES = np.round(np.arange(0.05, 0.951, 0.05), 2)
HIPER_POLARIDADE = {'taxa_aprendizagem': 2e-5, 'epocas': 3}
HIPER_EMOCOES = {'taxa_aprendizagem': 3e-5, 'epocas': 10}
COMUM = {'max_comprimento': 128, 'tamanho_lote': 16, 'decaimento_pesos': 0.01, 'proporcao_aquecimento': 0.1}

## 2. Dados

In [ ]:
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ''
    texto = re.sub(r'http\S+|www\.\S+', '', texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


URL_AFRISENTI = 'https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/data/por/{}.tsv'
POLARIDADE = {}
for split in ['train', 'dev', 'test']:
    df = pd.read_csv(URL_AFRISENTI.format(split), sep='\t', dtype=str)
    df['texto'] = df['tweet'].apply(limpar_texto)
    df['y'] = df['label'].map({s: i for i, s in enumerate(SENTIMENTOS)})
    assert df['y'].notna().all()
    POLARIDADE[split] = df.reset_index(drop=True)
    print('AfriSenti por', split, len(df))

from datasets import load_dataset
brighter = load_dataset('brighter-dataset/BRIGHTER-emotion-categories', 'ptmz')
EMO = {}
for split in ['train', 'dev', 'test']:
    df = brighter[split].to_pandas().drop_duplicates(subset='text').reset_index(drop=True)
    df['texto'] = df['text'].apply(limpar_texto)
    df[EMOCOES] = df[EMOCOES].astype(int)
    EMO[split] = df
    print('AfriEmo ptmz', split, len(df))
Y = {s: d[EMOCOES].values for s, d in EMO.items()}
positivos = Y['train'].sum(axis=0)
PESO_POSITIVO = np.sqrt((len(Y['train']) - positivos) / np.maximum(positivos, 1))
print('Pesos positivos:', dict(zip(EMOCOES, PESO_POSITIVO.round(2))))

## 3. Funções comuns

In [ ]:
class ConjuntoTexto(torch.utils.data.Dataset):
    def __init__(self, cod, rot):
        self.cod, self.rot = cod, rot

    def __len__(self):
        return len(self.rot)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.cod.items()}
        item['labels'] = self.rot[i]
        return item


class TrainerPonderado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        rotulos = inputs.pop('labels')
        saidas = model(**inputs)
        peso = torch.tensor(PESO_POSITIVO, dtype=saidas.logits.dtype, device=saidas.logits.device)
        perda = torch.nn.functional.binary_cross_entropy_with_logits(saidas.logits, rotulos.to(saidas.logits.dtype), pos_weight=peso)
        return (perda, saidas) if return_outputs else perda


def sigmoide(x):
    return 1 / (1 + np.exp(-x))


def calibrar_limiares(y, p):
    limiares = []
    for i in range(y.shape[1]):
        f1s = [f1_score(y[:, i], (p[:, i] >= t).astype(int), zero_division=0) for t in GRELHA_LIMIARES]
        limiares.append(float(GRELHA_LIMIARES[int(np.argmax(f1s))]))
    return np.array(limiares)


def argumentos(pasta, hiper):
    return TrainingArguments(
        output_dir=pasta, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
        learning_rate=hiper['taxa_aprendizagem'], per_device_train_batch_size=COMUM['tamanho_lote'],
        per_device_eval_batch_size=64, num_train_epochs=hiper['epocas'], weight_decay=COMUM['decaimento_pesos'],
        warmup_ratio=COMUM['proporcao_aquecimento'], fp16=True, seed=SEMENTE, data_seed=SEMENTE,
        logging_strategy='epoch', report_to='none', dataloader_num_workers=0)

## 4. Modelo de polaridade (XLM-R, português de Moçambique)

In [ ]:
set_seed(SEMENTE)
tok = AutoTokenizer.from_pretrained(ID_MODELO)
modelo = AutoModelForSequenceClassification.from_pretrained(ID_MODELO, num_labels=3,
    id2label=dict(enumerate(SENTIMENTOS)), label2id={s: i for i, s in enumerate(SENTIMENTOS)})


def preparar_pol(split):
    cod = tok(POLARIDADE[split]['texto'].tolist(), truncation=True, max_length=COMUM['max_comprimento'])
    return ConjuntoTexto(cod, POLARIDADE[split]['y'].astype(int).tolist())


def metrica_pol(ep):
    logits, y = ep
    return {'macro_f1': f1_score(y, logits.argmax(axis=1), average='macro', zero_division=0)}


t0 = time.time()
trainer = Trainer(model=modelo, args=argumentos(os.path.join(PASTA_TEMP, 'polaridade'), HIPER_POLARIDADE),
                  train_dataset=preparar_pol('train'), eval_dataset=preparar_pol('dev'),
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrica_pol)
trainer.train()
prev = trainer.predict(preparar_pol('test')).predictions.argmax(axis=1)
mf1_pol = f1_score(POLARIDADE['test']['y'], prev, average='macro')
print(f'Macro-F1 no teste: {100 * mf1_pol:.1f}%  (tempo: {(time.time() - t0) / 60:.1f} min)')

destino = os.path.join(PASTA_MODELOS, 'polaridade_pt_xlmr')
trainer.save_model(destino)
tok.save_pretrained(destino)
with open(os.path.join(destino, 'emotext.json'), 'w', encoding='utf-8') as f:
    json.dump({'tarefa': 'polaridade', 'lingua': 'português de Moçambique', 'modelo_base': ID_MODELO, 'semente': SEMENTE,
               'classes': SENTIMENTOS, 'macro_f1_teste': round(float(mf1_pol), 4), **HIPER_POLARIDADE, **COMUM},
              f, ensure_ascii=False, indent=2)
print('Guardado em', destino)
del trainer, modelo
torch.cuda.empty_cache()
shutil.rmtree(os.path.join(PASTA_TEMP, 'polaridade'), ignore_errors=True)

## 5. Modelo de emoções (XLM-R, português de Moçambique)

In [ ]:
set_seed(SEMENTE)
tok = AutoTokenizer.from_pretrained(ID_MODELO)
modelo = AutoModelForSequenceClassification.from_pretrained(ID_MODELO, num_labels=len(EMOCOES),
    problem_type='multi_label_classification', id2label=dict(enumerate(EMOCOES)), label2id={e: i for i, e in enumerate(EMOCOES)})


def preparar_emo(split):
    cod = tok(EMO[split]['texto'].tolist(), truncation=True, max_length=COMUM['max_comprimento'])
    return ConjuntoTexto(cod, [list(map(float, r)) for r in Y[split]])


def metrica_emo(ep):
    logits, y = ep
    p = sigmoide(logits)
    lim = calibrar_limiares(y.astype(int), p)
    return {'macro_f1': f1_score(y.astype(int), (p >= lim).astype(int), average='macro', zero_division=0)}


t0 = time.time()
trainer = TrainerPonderado(model=modelo, args=argumentos(os.path.join(PASTA_TEMP, 'emocoes'), HIPER_EMOCOES),
                           train_dataset=preparar_emo('train'), eval_dataset=preparar_emo('dev'),
                           data_collator=DataCollatorWithPadding(tok), compute_metrics=metrica_emo)
trainer.train()
limiares = calibrar_limiares(Y['dev'], sigmoide(trainer.predict(preparar_emo('dev')).predictions))
prev = (sigmoide(trainer.predict(preparar_emo('test')).predictions) >= limiares).astype(int)
mf1_emo = f1_score(Y['test'], prev, average='macro', zero_division=0)
print(f'Macro-F1 no teste: {100 * mf1_emo:.1f}%  (tempo: {(time.time() - t0) / 60:.1f} min)')
print('Limiares:', dict(zip(EMOCOES, limiares)))

destino = os.path.join(PASTA_MODELOS, 'emocoes_pt_xlmr')
trainer.save_model(destino)
tok.save_pretrained(destino)
with open(os.path.join(destino, 'emotext.json'), 'w', encoding='utf-8') as f:
    json.dump({'tarefa': 'emocoes', 'lingua': 'português de Moçambique', 'modelo_base': ID_MODELO, 'semente': SEMENTE,
               'emocoes': EMOCOES, 'limiares': dict(zip(EMOCOES, map(float, limiares))),
               'peso_positivo': dict(zip(EMOCOES, map(float, PESO_POSITIVO))), 'macro_f1_teste': round(float(mf1_emo), 4),
               **HIPER_EMOCOES, **COMUM}, f, ensure_ascii=False, indent=2)
print('Guardado em', destino)
del trainer, modelo
torch.cuda.empty_cache()
shutil.rmtree(os.path.join(PASTA_TEMP, 'emocoes'), ignore_errors=True)

## 6. Resumo

In [ ]:
for nome in ['polaridade_pt_xlmr', 'emocoes_pt_xlmr']:
    pasta = os.path.join(PASTA_MODELOS, nome)
    tamanho = sum(os.path.getsize(os.path.join(pasta, f)) for f in os.listdir(pasta)) / 1e6
    info = json.load(open(os.path.join(pasta, 'emotext.json'), encoding='utf-8'))
    print(f'{nome}: {tamanho:.0f} MB | Macro-F1 no teste = {100 * info["macro_f1_teste"]:.1f}%')
print('Descarregue a pasta EmoTextModelos do Google Drive para o computador.')